In [1]:
# import required classes
import pandas as pd
import glob
import re
from pathlib import Path
import numpy as np        
from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product
from tabulate import tabulate
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import cross_validate, cross_val_score, train_test_split, KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, accuracy_score, precision_score, recall_score, make_scorer, f1_score, precision_recall_curve, average_precision_score, roc_auc_score, roc_curve, auc
from sklearn.utils.class_weight import compute_class_weight
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer   
from sklearn.pipeline import Pipeline 
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# Classifiers
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

import os
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [2]:

print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.9.1
MPS available: True
CUDA available: False


In [3]:
#import data files
df = pd.read_csv('proccessed_dataset.csv', sep=',')

# Define numerical variables
numeric_cols = ['age_at_enrollment', 'total_credited', 'total_enrolled', 
'total_evaluations','total_approved', 'weighted_avg_grade', 'inflation_rate', 'gdp']


# Make lists for  categorical columns with decoded values
categorical_cols = ['marital_status_decoded', 'application_mode_decoded', 'course_decoded',
       'daytimeevening_attendance_decoded', 'previous_qualification_decoded', 'nationality_decoded',
       'mothers_qualification_decoded', 'fathers_qualification_decoded', 'mothers_occupation_decoded',
       'fathers_occupation_decoded', 'displaced_decoded', 'educational_special_needs_decoded',
       'debtor_decoded', 'tuition_fees_up_to_date_decoded', 'gender_decoded', 
       'scholarship_holder_decoded', 'international_decoded']

# convert col type to categorical
for i, col in enumerate(categorical_cols):
   df[col] = df[col].astype('category')


df['y'] = df['y'].astype('category')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3478 entries, 0 to 3477
Data columns (total 26 columns):
 #   Column                             Non-Null Count  Dtype   
---  ------                             --------------  -----   
 0   age_at_enrollment                  3478 non-null   int64   
 1   inflation_rate                     3478 non-null   float64 
 2   gdp                                3478 non-null   float64 
 3   marital_status_decoded             3478 non-null   category
 4   application_mode_decoded           3478 non-null   category
 5   course_decoded                     3478 non-null   category
 6   daytimeevening_attendance_decoded  3478 non-null   category
 7   previous_qualification_decoded     3478 non-null   category
 8   nationality_decoded                3478 non-null   category
 9   mothers_qualification_decoded      3478 non-null   category
 10  fathers_qualification_decoded      3478 non-null   category
 11  mothers_occupation_decoded         3478 non

In [4]:
# split our data
X = df.drop(['y'], axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=13, stratify=y)

In [5]:
# NN Using PyTorch
preprocessor_nn = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
            ('standardizer', StandardScaler())
        ]), numeric_cols)
    ], remainder='drop'
)


In [6]:
# preprocess training and testing set
X_train_prep = preprocessor_nn.fit_transform(X_train)
X_test_prep  = preprocessor_nn.transform(X_test)


In [7]:

# Convert to PyTorch tensors
X_train_clean = np.ascontiguousarray(X_train_prep, dtype=np.float32)
X_test_clean  = np.ascontiguousarray(X_test_prep, dtype=np.float32)

X_train_tensor = torch.from_numpy(X_train_clean)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1,1)

X_test_tensor = torch.from_numpy(X_test_clean)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).reshape(-1,1)

In [ ]:

os.environ["PYTORCH_MPS_DISABLE"] = "1"
device = torch.device("cpu")

class FlexibleNN(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(FlexibleNN, self).__init__()
        layers = []
        in_dim = input_size
        for h in hidden_units:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))  # Output layer
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def train_model(model, X_train, y_train, epochs=20, batch_size=32, lr=1e-3):
    model.to(device)
    model.train()
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
    return model


def evaluate_model(model, X, y):
    model.eval()
    with torch.no_grad():
        X = X.to(device)
        y = y.to(device)
        outputs = model(X)
        probs = torch.sigmoid(outputs).cpu().numpy().flatten()
        preds = (probs >= 0.5).astype(int)
        y_true = y.cpu().numpy().flatten()
    return {
        "accuracy": accuracy_score(y_true, preds),
        "precision": precision_score(y_true, preds, zero_division=0),
        "recall": recall_score(y_true, preds, zero_division=0),
        "f1": f1_score(y_true, preds, zero_division=0)
    }


def cross_validate_nn(X_tensor, y_tensor, hidden_units=[32], folds=5, epochs=20, batch_size=32, lr=1e-3):
    kf = KFold(n_splits=folds, shuffle=True, random_state=42)
    fold_metrics = []

    for train_idx, val_idx in kf.split(X_tensor):
        X_train_fold = X_tensor[train_idx]
        y_train_fold = y_tensor[train_idx]
        X_val_fold = X_tensor[val_idx]
        y_val_fold = y_tensor[val_idx]

        model = FlexibleNN(input_size=X_tensor.shape[1], hidden_units=hidden_units)
        train_model(model, X_train_fold, y_train_fold, epochs=epochs, batch_size=batch_size, lr=lr)
        metrics = evaluate_model(model, X_val_fold, y_val_fold)
        fold_metrics.append(metrics)

        # Free memory explicitly
        del model, X_train_fold, y_train_fold, X_val_fold, y_val_fold
        torch.cuda.empty_cache()
        import gc
        gc.collect()

    # Convert list of dicts to dict of lists
    all_metrics = {k: [m[k] for m in fold_metrics] for k in fold_metrics[0]}
    mean_metrics = {k: np.mean(v) for k, v in all_metrics.items()}
    std_metrics = {k: np.std(v) for k, v in all_metrics.items()}

    return mean_metrics, std_metrics


In [ ]:
# Define Simple NN with flexible number of hidden layers
class LogisticRegression(nn.Module):
    def __init__(self, input_size, hidden_units):
        super(LogisticRegression, self).__init__()
        
        in_dim = input_size
        layers = []
        for h in hidden_units:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.ReLU())
            in_dim = h

        layers.append(nn.Linear(in_dim, 1))

        self.network = nn.Sequential(*layers)


    def forward(self, x):
        return self.network(x)
